# Hindi Seq2Seq Summarisation with Attention

This notebook documents the custom bidirectional-LSTM encoder, additive-attention decoder, training, and inference workflow. It uses the official XL-Sum Hindi train/validation splits and the team's `01_Dataset/processed/common_test_set.csv` for final prediction. The shared test-set CSV must be populated with the agreed split before inference; this notebook will not silently substitute another test set.

Hindi text is tokenized with a Unicode-aware word/punctuation tokenizer. Vocabulary is built from training rows only and includes PAD, UNK, BOS, and EOS tokens. No pretrained tokenizer or model is required.

A CUDA GPU is recommended. Full training is compute-intensive; pass sample limits for a smoke run, not for final reported results.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / '03_Source_Code' / 'seq2seq').is_dir():
    ROOT = ROOT.parent
assert (ROOT / '03_Source_Code' / 'seq2seq').is_dir(), 'Open this notebook from the project root or 02_Notebooks.'
CODE = ROOT / '03_Source_Code' / 'seq2seq'
print('Project root:', ROOT)
print('Python:', sys.executable)

## Training

The default settings train on the complete official training split for up to 10 epochs and save the best validation-loss checkpoint under `04_Models/seq2seq_attention/checkpoints/`. The command below uses the defaults. For a short pipeline check, add `--max-train-samples 128 --max-validation-samples 32 --epochs 1`; do not present those smoke-run metrics as final.

In [ ]:
train_command = [sys.executable, str(CODE / 'train.py')]
# Example smoke-test options (remove/comment these for the agreed full experiment):
# train_command += ['--max-train-samples', '128', '--max-validation-samples', '32', '--epochs', '1']
print(' '.join(train_command))
subprocess.run(train_command, cwd=ROOT, check=True)

Training settings are saved beside the checkpoint as `training_settings.json`; each epoch's token-weighted train and validation cross-entropy losses are recorded in `training_history.csv`. The best checkpoint is selected by validation loss.

## Prediction and ROUGE

Inference requires the agreed shared test file with `article_id`, `reference`, and either `article` or `text` columns. The output follows the shared `article_id,reference,prediction` schema. The script prints corpus-average ROUGE-1, ROUGE-2, and ROUGE-L precision, recall, and F-measure and writes them to `test_rouge_scores.csv` beside the checkpoint. Keep only the full agreed test-set run in final project results.

In [ ]:
test_file = ROOT / '01_Dataset' / 'processed' / 'common_test_set.csv'
if not test_file.is_file() or test_file.stat().st_size == 0:
    raise FileNotFoundError(f'Populate the agreed shared test set before prediction: {test_file}')
predict_command = [sys.executable, str(CODE / 'predict.py')]
print(' '.join(predict_command))
subprocess.run(predict_command, cwd=ROOT, check=True)